In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("LakehouseDebug")
    .config("spark.python.worker.faulthandler.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Worker Python:", spark.sparkContext.pythonExec)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/25 17:30:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Worker Python: /usr/local/bin/python


In [2]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, DoubleType

from pathlib import Path

cwd = Path.cwd()

project_root = (
    cwd.parent
    if cwd.name == "notebooks"
    else cwd
)

path = project_root / "data" / "raw" / "sales_orders.csv"

#  If we wanted to infer schema we would
#  df = spark.read.csv(path, header=True, inferSchema=True)


''' Correct Schema 
schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("line_id", IntegerType(), True),
    StructField("order_date", DateType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("order_status", StringType(), True),
])'''

schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("line_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("unit_price", StringType(), True),
    StructField("order_status", StringType(), True),
])

df = spark.read.csv(str(path), header=True, schema=schema)
print(df)

DataFrame[order_id: string, line_id: string, order_date: string, customer_id: string, product_id: string, quantity: string, unit_price: string, order_status: string]


In [3]:
import uuid
from pyspark.sql import functions as F

batch_id = str(uuid.uuid4())

# Here we use Spark specific functions rather than PY
bronze_df = (
    df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit(str(path)))
    .withColumn("_batch_id", F.lit(batch_id))
)
print(bronze_df)

DataFrame[order_id: string, line_id: string, order_date: string, customer_id: string, product_id: string, quantity: string, unit_price: string, order_status: string, _ingested_at: timestamp, _source_file: string, _batch_id: string]


In [4]:
bronze_df.printSchema()
bronze_df.show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- line_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = false)
 |-- _source_file: string (nullable = false)
 |-- _batch_id: string (nullable = false)

+--------+-------+----------+-----------+----------+--------+----------+------------+-------------------------+------------------------------------+------------------------------------+
|order_id|line_id|order_date|customer_id|product_id|quantity|unit_price|order_status|_ingested_at             |_source_file                        |_batch_id                           |
+--------+-------+----------+-----------+----------+--------+----------+------------+-------------------------+------------------------------------+-------

In [5]:
assert bronze_df.count() == df.count()
assert len(bronze_df.columns) == 11
assert bronze_df.select("_batch_id").distinct().count() == 1

# Once asserted
write_path = project_root / "data" / "bronze" / "sales_orders"

bronze_df.write.mode("overwrite").parquet(str(write_path))